In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from math import radians, cos, sin, asin, sqrt


In [ ]:

# --- 1. Load the Data ---
df = pd.read_csv('data/ski_resorts.csv')

In [ ]:
# --- 2. Clean the Data ---
# Drop any rows where critical location data is missing
df = df.dropna(subset=['latitude', 'longitude'])

# --- 3. The Distance Logic (Haversine Formula) ---
def calculate_distance(lat1, lon1, lat2, lon2):
    """Calculates the distance in miles between two GPS coordinates."""
    # Convert decimal degrees to radians
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    
    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    
    # Radius of earth in miles is ~3956
    return c * 3956

# Rochester, NY coordinates
roc_lat, roc_lon = 43.16, -77.61
# Apply the function to create a new column
df['distance_miles'] = df.apply(
    lambda row: calculate_distance(roc_lat, roc_lon, row['latitude'], row['longitude']), 
    axis=1
)

In [ ]:
# --- 4. Filter and Visualize ---
# Assuming an average driving speed of 50mph, 5 hours is roughly 250 miles
nearby_resorts = df[df['distance_miles'] <= 250].copy()

# Sort by size to find the "best" (using vertical drop as a proxy for size)
nearby_resorts = nearby_resorts.sort_values(by='vertical_drop_ft', ascending=False)

print(nearby_resorts[['resort_name', 'distance_miles', 'vertical_drop_ft']])

# Quick visualization
plt.figure(figsize=(10, 5))
plt.bar(nearby_resorts['resort_name'], nearby_resorts['vertical_drop_ft'], color='skyblue')
plt.title('Vertical Drop of Ski Resorts Within 5 Hours of Rochester, NY')
plt.ylabel('Vertical Drop (Feet)')
plt.show()